In [ ]:
# ==============================
# Cell 2 — Imports and settings
# ==============================
import re
import time
import warnings
from io import BytesIO
from pathlib import Path
from urllib.parse import urlencode
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from tqdm.auto import tqdm

from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.ipac.irsa import Irsa

warnings.filterwarnings("ignore")

# ----------------------------
# Configuration
# ----------------------------
N_PROTOTYPE = 3
CUTOUT_SIZE_ARCSEC = 10.0
SEARCH_RADIUS_DEG = 0.4

COSMOS_RA_CENTER = 150.1192
COSMOS_DEC_CENTER = 2.2058

CATALOG_NAME = "acs_iphot_sep07"
CUTOUT_SERVICE_URL = "https://irsa.ipac.caltech.edu/cgi-bin/Cutouts/nph-cutouts"

OUT_DIR = Path("cosmos_cutouts")
RAW_DIR = OUT_DIR / "raw_fits"
NPY_DIR = OUT_DIR / "npy"
PREVIEW_DIR = OUT_DIR / "preview"
XML_DIR = OUT_DIR / "xml"

for d in [OUT_DIR, RAW_DIR, NPY_DIR, PREVIEW_DIR, XML_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(42)

session = requests.Session()
session.headers.update({"User-Agent": "cosmos-cutout-extraction/1.0"})

print("Output directory:", OUT_DIR.resolve())
print("Requested final sample size:", N_PROTOTYPE)
print("Cutout size (arcsec):", CUTOUT_SIZE_ARCSEC)
print("Catalog:", CATALOG_NAME)

In [ ]:
# ==========================================
# Cell 3 — Query the COSMOS ACS source table
# ==========================================
center = SkyCoord(ra=COSMOS_RA_CENTER * u.deg, dec=COSMOS_DEC_CENTER * u.deg)

requested_columns = [
    "ra",
    "dec",
    "mu_class",
    "mag_auto",
    "flux_radius",
    "elongation",
    "flags",
    "a_world",
    "b_world",
]

table_info = Irsa.list_columns(CATALOG_NAME)
available_columns = list(table_info.keys())
valid_columns = [c for c in requested_columns if c in available_columns]
missing_columns = [c for c in requested_columns if c not in available_columns]

print("Valid columns:", valid_columns)
if missing_columns:
    print("Missing columns:", missing_columns)

if not {"ra", "dec"}.issubset(valid_columns):
    raise ValueError("Catalog must contain ra and dec columns.")

cat_table = Irsa.query_region(
    coordinates=center,
    catalog=CATALOG_NAME,
    spatial="Cone",
    radius=SEARCH_RADIUS_DEG * u.deg,
    columns=",".join(valid_columns),
)

cat = cat_table.to_pandas()
print(f"Raw catalog rows: {len(cat):,}")
cat.head()

In [ ]:
# ==================================
# Cell 4 — Clean and quality-filter
# ==================================
work = cat.copy()

for col in ["ra", "dec", "mu_class", "mag_auto", "flux_radius", "elongation", "flags", "a_world", "b_world"]:
    if col in work.columns:
        work[col] = pd.to_numeric(work[col], errors="coerce")

# Base validity
work = work.replace([np.inf, -np.inf], np.nan)
work = work.dropna(subset=["ra", "dec"])

# Galaxy-only, good detections, physically reasonable morphology
mask = np.ones(len(work), dtype=bool)

if "mu_class" in work.columns:
    mask &= (work["mu_class"] == 1)

if "flags" in work.columns:
    mask &= (work["flags"].fillna(99) == 0)

if "mag_auto" in work.columns:
    mask &= work["mag_auto"].between(18.0, 25.5)

if "flux_radius" in work.columns:
    mask &= work["flux_radius"].between(1.0, 20.0)

if "elongation" in work.columns:
    mask &= work["elongation"].between(1.0, 4.5)

if "a_world" in work.columns:
    mask &= work["a_world"].fillna(-1) > 0

if "b_world" in work.columns:
    mask &= work["b_world"].fillna(-1) > 0

filtered = work.loc[mask].copy()
filtered = filtered.dropna(subset=["mag_auto", "flux_radius", "elongation"]).reset_index(drop=True)
filtered["axis_ratio_est"] = 1.0 / filtered["elongation"]

print(f"Rows after cuts: {len(filtered):,}")
filtered[["mag_auto", "flux_radius", "elongation", "axis_ratio_est"]].describe().round(3)

In [ ]:
# ==========================================
# Cell 5 — Morphology-aware diverse sampling
# ==========================================
def safe_qcut(series, q, labels):
    n_unique = series.nunique(dropna=True)
    q_eff = min(q, n_unique)
    if q_eff < 2:
        return pd.Series([labels[0]] * len(series), index=series.index)
    return pd.qcut(series, q=q_eff, labels=labels[:q_eff], duplicates="drop")

tmp = filtered.copy()

tmp["size_bin"] = safe_qcut(tmp["flux_radius"], 4, ["size_1", "size_2", "size_3", "size_4"])
tmp["mag_bin"] = safe_qcut(tmp["mag_auto"], 4, ["mag_1", "mag_2", "mag_3", "mag_4"])
tmp["shape_bin"] = safe_qcut(tmp["elongation"], 4, ["shape_1", "shape_2", "shape_3", "shape_4"])

diverse_seed = []
for _, grp in tmp.groupby(["size_bin", "mag_bin", "shape_bin"], observed=True):
    if len(grp) > 0:
        diverse_seed.append(grp.sample(n=1, random_state=int(RNG.integers(0, 1_000_000_000))))

diverse_seed = pd.concat(diverse_seed, ignore_index=True).drop_duplicates(subset=["ra", "dec"])
remainder = tmp.drop(diverse_seed.index, errors="ignore").sample(frac=1, random_state=42)

candidate_pool = pd.concat([diverse_seed, remainder], ignore_index=True).drop_duplicates(subset=["ra", "dec"])
candidate_pool = candidate_pool.reset_index(drop=True)

candidate_pool["source_id"] = [f"cosmos_{i:05d}" for i in range(len(candidate_pool))]

print("Diverse seed size:", len(diverse_seed))
print("Candidate pool size:", len(candidate_pool))
candidate_pool.head()

In [ ]:
# ===========================================
# Cell 6 — Helpers for IRSA cutout retrieval
# ===========================================
def request_cutout_manifest(ra, dec, size_arcsec):
    params = {
        "mission": "COSMOS",
        "locstr": f"{ra} {dec} eq j2000",
        "sizeX": size_arcsec,
        "sizeY": size_arcsec,
        "units": "arcsec",
        "mode": "PI",
    }
    r = session.get(CUTOUT_SERVICE_URL, params=params, timeout=120)
    r.raise_for_status()
    return r.text, r.url

def extract_all_urls(text):
    urls = []
    try:
        root = ET.fromstring(text)
        for elem in root.iter():
            if elem.text and isinstance(elem.text, str) and elem.text.strip().startswith("http"):
                urls.append(elem.text.strip())
            for v in elem.attrib.values():
                if isinstance(v, str) and v.startswith("http"):
                    urls.append(v.strip())
    except ET.ParseError:
        pass

    regex_urls = re.findall(r"https?://[^\s<>'\"]+", text)
    urls.extend(regex_urls)

    seen = set()
    unique = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            unique.append(u)
    return unique

def score_fits_url(url):
    u = url.lower()
    score = 0
    if ".fits" in u:
        score += 100
    if "acs" in u or "hst" in u:
        score += 25
    if "f814" in u or "f814w" in u:
        score += 25
    if "cutout" in u:
        score += 5
    return score

def choose_best_fits_url(urls):
    fits_urls = [u for u in urls if ".fits" in u.lower()]
    if not fits_urls:
        return None
    fits_urls = sorted(fits_urls, key=score_fits_url, reverse=True)
    return fits_urls[0]

def download_best_cutout(ra, dec, size_arcsec, xml_path, fits_path):
    xml_text, request_url = request_cutout_manifest(ra, dec, size_arcsec)
    xml_path.write_text(xml_text, encoding="utf-8")

    urls = extract_all_urls(xml_text)
    best_fits = choose_best_fits_url(urls)

    if best_fits is None:
        raise RuntimeError("No FITS URL found in IRSA cutout response.")

    rr = session.get(best_fits, timeout=120)
    rr.raise_for_status()
    fits_path.write_bytes(rr.content)

    return {
        "request_url": request_url,
        "fits_url": best_fits,
        "all_urls": urls,
    }

def read_first_2d_image(fits_path):
    with fits.open(fits_path, memmap=False) as hdul:
        for hdu in hdul:
            data = getattr(hdu, "data", None)
            if data is None:
                continue
            arr = np.asarray(data)
            arr = np.squeeze(arr)
            if arr.ndim == 2:
                return arr.astype(np.float32)
    raise ValueError("No 2D image plane found in FITS file.")

def robust_normalize(image):
    img = np.asarray(image, dtype=np.float32).copy()
    finite = np.isfinite(img)
    if finite.sum() == 0:
        return np.zeros_like(img, dtype=np.float32)

    fill = np.nanmedian(img[finite])
    img[~finite] = fill

    # background subtract
    img = img - np.nanmedian(img)

    p1, p99 = np.nanpercentile(img, [1, 99])
    if not np.isfinite(p1) or not np.isfinite(p99) or p99 <= p1:
        p1, p99 = np.nanmin(img), np.nanmax(img)

    if not np.isfinite(p1) or not np.isfinite(p99) or p99 <= p1:
        return np.zeros_like(img, dtype=np.float32)

    img = np.clip((img - p1) / (p99 - p1), 0.0, 1.0)
    return img.astype(np.float32)

In [ ]:
# ===========================================================
# Cell 7 — Download until 50 successful normalized cutouts
# ===========================================================
records = []
success_rows = []

for _, row in tqdm(candidate_pool.iterrows(), total=len(candidate_pool)):
    if len(success_rows) >= N_PROTOTYPE:
        break

    source_id = row["source_id"]
    ra = float(row["ra"])
    dec = float(row["dec"])

    xml_path = XML_DIR / f"{source_id}.xml"
    fits_path = RAW_DIR / f"{source_id}.fits"
    npy_path = NPY_DIR / f"{source_id}.npy"
    png_path = PREVIEW_DIR / f"{source_id}.png"

    try:
        info = download_best_cutout(
            ra=ra,
            dec=dec,
            size_arcsec=CUTOUT_SIZE_ARCSEC,
            xml_path=xml_path,
            fits_path=fits_path,
        )

        raw = read_first_2d_image(fits_path)
        norm = robust_normalize(raw)

        if norm.ndim != 2 or min(norm.shape) < 16:
            raise ValueError(f"Unexpected cutout shape: {norm.shape}")

        np.save(npy_path, norm)
        plt.imsave(png_path, norm, cmap="gray", origin="lower")

        rec = row.to_dict()
        rec.update({
            "status": "ok",
            "request_url": info["request_url"],
            "fits_url": info["fits_url"],
            "fits_path": str(fits_path),
            "npy_path": str(npy_path),
            "png_path": str(png_path),
            "image_shape": tuple(norm.shape),
        })
        records.append(rec)
        success_rows.append(rec)

    except Exception as e:
        rec = row.to_dict()
        rec.update({
            "status": "failed",
            "error": str(e),
            "fits_path": str(fits_path),
            "npy_path": str(npy_path),
        })
        records.append(rec)

    time.sleep(0.2)

results = pd.DataFrame(records)
results.to_csv(OUT_DIR / "download_attempts.csv", index=False)

success_df = pd.DataFrame(success_rows).reset_index(drop=True)
success_df.to_csv(OUT_DIR / "selected_sources_metadata.csv", index=False)

print("Successful cutouts:", len(success_df))
print("Failed attempts:", (results["status"] == "failed").sum())
display(success_df.head())

In [ ]:
# ===========================================
# Cell 8 — Sanity checks on final sample set
# ===========================================
if len(success_df) < N_PROTOTYPE:
    print(f"Warning: only {len(success_df)} successful cutouts were produced.")
else:
    print(f"Success: produced {len(success_df)} normalized galaxy cutouts.")

summary_cols = [c for c in ["mag_auto", "flux_radius", "elongation", "axis_ratio_est"] if c in success_df.columns]
display(success_df[summary_cols].describe().round(3))

In [ ]:
# ======================================
# Cell 9 — Show a quick preview mosaic
# ======================================
n_show = min(25, len(success_df))
if n_show == 0:
    raise RuntimeError("No successful cutouts to display.")

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
axes = axes.ravel()

for ax in axes:
    ax.axis("off")

for i in range(n_show):
    row = success_df.iloc[i]
    img = np.load(row["npy_path"])
    axes[i].imshow(img, cmap="gray", origin="lower")
    axes[i].set_title(row["source_id"], fontsize=8)
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Cell 10 — Final compact manifest for later
# ============================================
manifest_cols = [
    "source_id",
    "ra",
    "dec",
    "mag_auto",
    "flux_radius",
    "elongation",
    "axis_ratio_est",
    "fits_url",
    "fits_path",
    "npy_path",
    "png_path",
]

final_manifest = success_df[[c for c in manifest_cols if c in success_df.columns]].copy()
final_manifest.to_csv(OUT_DIR / "final_manifest.csv", index=False)

print("Saved:")
print(" -", OUT_DIR / "download_attempts.csv")
print(" -", OUT_DIR / "selected_sources_metadata.csv")
print(" -", OUT_DIR / "final_manifest.csv")
print(" -", NPY_DIR)
print(" -", PREVIEW_DIR)